# Flexible edge orientation

Some edges shouldn't be statically directed. Their orientation depends
on the current value of an attribute — a temperature gradient flips a
flow direction; a regulatory signal switches an activation to an
inhibition; a measurement crosses a threshold.

AnnNet supports this through *edge direction policies*: declare the
policy at edge creation, then update the watched attribute later and
the incidence column flips automatically.


## Anatomy of a policy

A policy dict has these keys:

| key | meaning |
|---|---|
| `var` | name of the attribute to watch |
| `threshold` | the cut-off value |
| `scope` | `'edge'` (watch an edge attribute) or `'vertex'` (watch a vertex attribute on src/tgt) |
| `above` | which direction wins when `x > threshold`: `'s->t'` or `'t->s'` |
| `tie` | what to do at equality: `'keep'`, `'undirected'`, `'s->t'`, `'t->s'` |


## 1. Edge-scope policy

Watch an edge attribute. Above the threshold ⇒ orient src→tgt;
below ⇒ orient tgt→src.


In [ ]:
from annnet import AnnNet

G = AnnNet(directed=True)
G.add_vertices(['A', 'B'])

# Edge AB whose orientation is governed by its 'temperature' attribute.
G.add_edges(
    'A',
    'B',
    edge_id='heat_flow',
    weight=1.0,
    flexible={
        'var': 'temperature',
        'threshold': 10.0,
        'scope': 'edge',
        'above': 's->t',
        'tie': 'keep',
    },
)
print('policy registered on:', list(G.edge_direction_policy))

In [ ]:
# Helper to inspect the incidence column.
def show_column(graph, eid):
    rec = graph._edges[eid]
    col = rec.col_idx
    M = graph._matrix
    print(f'incidence column for {eid}:')
    for vid in graph.vertices():
        row = graph._entities[graph._resolve_entity_key(vid)].row_idx
        v = M[row, col]
        print(f'  {vid}: {v:+.2f}')


show_column(G, 'heat_flow')

In [ ]:
# Push temperature above the threshold → orient src→tgt (+w on A, -w on B).
G.attrs.set_edge_attrs('heat_flow', temperature=20.0)
show_column(G, 'heat_flow')

In [ ]:
# Push temperature below the threshold → orient tgt→src (-w on A, +w on B).
G.attrs.set_edge_attrs('heat_flow', temperature=5.0)
show_column(G, 'heat_flow')

## 2. Vertex-scope policy

Watch a vertex attribute on src and tgt. The condition fires when
`xs - xt > 0` (or `< 0`, depending on `above`).


In [ ]:
H = AnnNet(directed=True)
H.add_vertices(['A', 'B'])
H.add_edges(
    'A',
    'B',
    edge_id='diffusion',
    weight=1.0,
    flexible={
        'var': 'concentration',
        'threshold': 0.0,  # threshold on the difference (xs - xt)
        'scope': 'vertex',
        'above': 's->t',
        'tie': 'keep',
    },
)

# A high, B low → expect s->t orientation
H.attrs.set_vertex_attrs('A', concentration=10.0)
H.attrs.set_vertex_attrs('B', concentration=2.0)
show_column(H, 'diffusion')

In [ ]:
# Flip the gradient → expect t->s orientation
H.attrs.set_vertex_attrs('A', concentration=1.0)
H.attrs.set_vertex_attrs('B', concentration=9.0)
show_column(H, 'diffusion')

## 3. Tie handling

When the watched value equals the threshold, the `tie` setting decides
the outcome.

- `'keep'` — do nothing (incidence stays as it was)
- `'undirected'` — force `+w` on both endpoints
- `'s->t'` — force src→tgt
- `'t->s'` — force tgt→src


In [ ]:
K = AnnNet(directed=True)
K.add_vertices(['A', 'B'])

for tie_mode in ('keep', 'undirected', 's->t', 't->s'):
    K.remove_edges(list(K._edges), errors='ignore')
    K.add_edges(
        'A',
        'B',
        edge_id=f'tie_{tie_mode}',
        weight=1.0,
        flexible={'var': 'x', 'threshold': 5.0, 'scope': 'edge', 'tie': tie_mode},
    )
    # Set x exactly at the threshold.
    K.attrs.set_edge_attrs(f'tie_{tie_mode}', x=5.0)
    print(f'tie={tie_mode}:')
    show_column(K, f'tie_{tie_mode}')
    print()

## 4. Bulk attribute changes

A bulk call (`set_vertex_attrs_bulk`, `set_edge_attrs_bulk`) collects
all the policies that need re-evaluation and applies them once at the
end. No need to call any orientation method by hand.


In [ ]:
B = AnnNet(directed=True)
B.add_vertices(['A', 'B', 'C'])
B.add_edges(
    'A',
    'B',
    edge_id='ab',
    weight=1.0,
    flexible={'var': 'level', 'threshold': 5.0, 'scope': 'vertex'},
)
B.add_edges(
    'B',
    'C',
    edge_id='bc',
    weight=1.0,
    flexible={'var': 'level', 'threshold': 5.0, 'scope': 'vertex'},
)

B.attrs.set_vertex_attrs_bulk(
    {
        'A': {'level': 10.0},
        'B': {'level': 8.0},
        'C': {'level': 1.0},
    }
)

for eid in ('ab', 'bc'):
    show_column(B, eid)
    print()

## When to use this

- Signal-direction switching driven by phosphorylation state
- Flow direction driven by a measured gradient
- Conditional activation / inhibition based on a regulator's concentration

For static directed edges, just pass `directed=True` to `add_edges` — no
policy needed.
